# 04 — Comparacion Etapa 2

Compara E2-A (CCI+SMA+Vol+ATR) vs E2-C (SMA+RSI+MACD) vs Buy & Hold. Long-only, metricas con fees+slippage.

In [ ]:
# 04 — Comparacion Etapa 2 (fiel a notebooks/06): E2-A vs E2-C vs Buy & Hold — long-only
import polars as pl, yaml
from src.features import build_features
from src.strategies.e2a_composite import signal_e2a_long, E2A_A1, E2A_A2_BEST
from src.strategies.e2c_winners import signal_e2c_long
from src.strategies.buy_and_hold import signal_buy_and_hold
from src.backtesting.engine import backtest
from src.backtesting.metrics import summarize
from src.evaluation.statistics import compare_strategies
from src.evaluation.plots import plot_equity
cfg = yaml.safe_load(open("../configs/strategies.yaml"))
df = build_features(pl.read_parquet("../data/processed/btcusdt_1m.parquet"))
# 1) Mezcla CCI+SMA+Volumen con ATR stop dinamico (A1 paper; cambiar a E2A_A2_BEST para optimizado)
df_e2a = signal_e2a_long(df, **E2A_A1)
# 2) Ganadoras etapa 1: SMA regimen + MACD trigger + RSI filtro
df_e2c = signal_e2c_long(df, **cfg["strategies"]["e2c_winners"]["params"])
# 3) Hold BTC con el capital (benchmark)
df_bh = signal_buy_and_hold(df)
bt_e2a = backtest(df_e2a, signal_col="signal_e2a", fee_bps=10, slippage_bps=5, lag=1)
bt_e2c = backtest(df_e2c, signal_col="signal_e2c", fee_bps=10, slippage_bps=5, lag=1)
bt_bh = backtest(df_bh, signal_col="signal_bh", fee_bps=10, slippage_bps=5, lag=1)
print("E2-A:", summarize(bt_e2a))
print("E2-C:", summarize(bt_e2c))
print("BH:", summarize(bt_bh))
compare_strategies({"e2a_composite": bt_e2a, "e2c_winners": bt_e2c, "buy_and_hold": bt_bh})

In [ ]:
plot_equity(bt_trend, title="Trend Following — Equity")
plot_equity(bt_mr, title="Mean Reversion — Equity")